# Phase 14: codec-aware DCT masking pilot (faculty item 7)

Continues the Phase 11 dual-stream (DFDC held out) for 5 epochs in two matched arms: **control** (plain fine-tuning) and **masked** (JPEG-table-guided block-DCT masking on the frequency-branch input). It then scores DFDC clean, JPEG q50 and JPEG q20. Main result: masked minus control at JPEG q20, with a paired 95% interval.

**Import it into a Kaggle notebook** (File, Import Notebook) with these inputs attached: the three face-crop datasets and `phase11-checkpoints`. Settings: GPU (one GPU is used), Internet not needed.

1. Run Cells 1 to 5 interactively (about 10 to 15 min). Cell 5 must finish its unit tests, data audit and real-model preflight without an error.
2. Then Save Version, Save & Run All (Commit). Expect about 1 to 2 h.
3. Download the zip from Cell 7 (or the version's Output tab) into the Download folder.

Cells 1 and 2 are the versions that ran Phase 13b (known dataset paths, `torch.autocast` fix). Cell 3 points at the uploaded checkpoint directly. Cells 4 to 7 are the September pilot, unchanged.

## Cell 1 — Your original setup, with TERM=0

TERM=0 prevents the old TERM=11/12 experiment blocks from launching. No original training is started here.

In [ ]:
import os, re, json, time, random, warnings, itertools
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
from sklearn.model_selection import GroupShuffleSplit
from sklearn.metrics import roc_auc_score, roc_curve, f1_score, accuracy_score, \
                            precision_score, recall_score, confusion_matrix
from PIL import Image
import timm

warnings.filterwarnings("ignore")

# ===================== CONFIG ========================================
TERM  = 0    # 1=audit(no GPU) 2=matrix+LOO 3=GradCAM 4=robustness 11=dual/DANN/CAM
SMOKE = False    # False = the real run

SEED, IMG_SIZE, BATCH = 42, 224, 64
LR, WD, NUM_WORKERS = 1e-4, 1e-4, 4
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

if SMOKE:
    EPOCHS, PATIENCE = 1, 1
    MAX_IMGS_PER_DATASET, MAX_TEST_IMGS, MAX_VAL_IMGS = 3000, 3000, 1500
else:
    EPOCHS, PATIENCE = 20, 4
    MAX_IMGS_PER_DATASET, MAX_TEST_IMGS, MAX_VAL_IMGS = 30000, None, None

OUT = Path("/kaggle/working/phase5_results"); OUT.mkdir(parents=True, exist_ok=True)
PHASE4_DIR = Path("/kaggle/input/phase4-results")

random.seed(SEED); np.random.seed(SEED)
torch.manual_seed(SEED); torch.cuda.manual_seed_all(SEED)

# ===================== DISCOVERY =====================================
INPUT_ROOT = Path("/kaggle/input")

def _find_root(*hints, max_depth=5):
    cands = []
    for dp, dns, fns in os.walk(INPUT_ROOT):
        rel = os.path.relpath(dp, INPUT_ROOT).lower()
        if rel == ".": continue
        if rel.count(os.sep) + 1 > max_depth:
            dns[:] = []; continue
        if any(h in rel for h in hints):
            cands.append((rel.count(os.sep) + 1, dp)); dns[:] = []
    return Path(min(cands)[1]) if cands else None

# Known paths first: searching /kaggle/input takes up to an hour in a commit.
_KNOWN = {"celebdf": "/kaggle/input/datasets/amanrawat001/celeb-df-preprocessed",
          "ffpp":    "/kaggle/input/datasets/debajyatidey/faceforensics-videos-cropped-faces",
          "dfdc":    "/kaggle/input/datasets/itamargr/dfdc-faces-of-the-train-sample"}
def _root(name, *hints):
    p = Path(_KNOWN[name])
    return p if p.is_dir() else _find_root(*hints)
CELEBDF_ROOT = _root("celebdf", "celebdf", "celeb-df", "celeb_df")
FFPP_ROOT    = _root("ffpp", "faceforensics", "ff++", "ffpp", "forensics")
DFDC_ROOT    = _root("dfdc", "dfdc")
print("Celeb-DF:", CELEBDF_ROOT, "\nFF++    :", FFPP_ROOT, "\nDFDC    :", DFDC_ROOT)

IMG_EXT = (".jpg", ".jpeg", ".png", ".bmp", ".webp")
CLASS_DIRS = {"fake","fakes","real","reals","original","originals","manipulated",
              "manipulated_sequences","original_sequences","pristine","youtube",
              "train","val","valid","validation","test"}

def _walk_images(root):
    for dp, _, fns in os.walk(root):
        for fn in fns:
            if fn.lower().endswith(IMG_EXT):
                yield os.path.join(dp, fn)

def _label_from_path(path):
    for p in reversed([x.lower() for x in Path(path).parts]):
        if p in ("fake","fakes","manipulated","manipulated_sequences","deepfake"): return 1
        if p in ("real","reals","original","originals","pristine","youtube",
                 "original_sequences"): return 0
    return None

_FRAME_RE = re.compile(r"^(.*?)[_-]?frame[_-]?\d+", re.IGNORECASE)

def vid_celebdf(path):
    stem = Path(path).stem
    m = _FRAME_RE.match(stem)
    return m.group(1) if (m and m.group(1)) else "_".join(stem.split("_")[:3])

def vid_ffpp(path):
    """fake/588_556_crops/frame_0019.png -> the FOLDER is the video."""
    p = Path(path)
    return p.parent.name if p.parent.name.lower() not in CLASS_DIRS else p.stem

def vid_dfdc(path):
    """aapnvogymq_100_1.png -> aapnvogymq"""
    return Path(path).stem.split("_")[0]

PARSERS = {"celebdf": (CELEBDF_ROOT, vid_celebdf),
           "ffpp":    (FFPP_ROOT,    vid_ffpp),
           "dfdc":    (DFDC_ROOT,    vid_dfdc)}

def _ffpp_identity_groups(videos):
    """Tie fake 588_556 to real 588 and real 556 — same face, same split."""
    parent = {}
    def find(x):
        parent.setdefault(x, x)
        while parent[x] != x:
            parent[x] = parent[parent[x]]; x = parent[x]
        return x
    def union(a, b):
        ra, rb = find(a), find(b)
        if ra != rb: parent[ra] = rb
    nums_of = {}
    for v in videos:
        nums = re.findall(r"\d+", v.split("::")[-1])
        nums_of[v] = nums
        for n in nums[1:]: union(nums[0], n)
    return {v: (f"ffppid::{find(ns[0])}" if ns else v) for v, ns in nums_of.items()}

def build_manifest(name):
    root, vid_fn = PARSERS[name]
    assert root is not None, f"{name} not attached"
    rows = []
    for p in _walk_images(root):
        y = _label_from_path(p)
        if y is None: continue
        rows.append((p, y, f"{name}::{vid_fn(p)}", name))
    df = pd.DataFrame(rows, columns=["path","label","video","dataset"])
    assert len(df) > 0, f"No labelled images under {root}"
    if name == "ffpp":
        df["group"] = df.video.map(_ffpp_identity_groups(df.video.unique()))
        print(f"  [ffpp] {df.video.nunique():,} videos -> {df.group.nunique():,} identity groups")
    else:
        df["group"] = df.video
    return df

def split_by_video(df, test_size=0.15, val_size=0.15, seed=SEED):
    n = df.group.nunique()
    assert n >= 20, f"Only {n} groups — parser broken: {df.group.head(3).tolist()}"
    gss = GroupShuffleSplit(1, test_size=test_size, random_state=seed)
    tr_idx, te_idx = next(gss.split(df, groups=df.group))
    trv, te = df.iloc[tr_idx].reset_index(drop=True), df.iloc[te_idx].reset_index(drop=True)
    gss2 = GroupShuffleSplit(1, test_size=val_size/(1-test_size), random_state=seed)
    tr_i2, va_i = next(gss2.split(trv, groups=trv.group))
    tr, va = trv.iloc[tr_i2].reset_index(drop=True), trv.iloc[va_i].reset_index(drop=True)
    for a, b, nm in [(tr,va,"train/val"), (tr,te,"train/test"), (va,te,"val/test")]:
        for col in ("group", "video"):
            ov = set(a[col]) & set(b[col])
            assert len(ov) == 0, f"LEAK in {nm} on {col}: {len(ov)} shared"
    return tr, va, te

def cap_per_dataset(df, cap=None, seed=SEED):
    cap = MAX_IMGS_PER_DATASET if cap is None else cap
    out = [g.sample(min(len(g), cap), random_state=seed) if len(g) > cap else g
           for _, g in df.groupby("dataset")]
    return pd.concat(out).sample(frac=1.0, random_state=seed).reset_index(drop=True)

def subsample(df, cap, seed=SEED):
    """Shrink a split by dropping whole VIDEOS, never single frames."""
    if cap is None or len(df) <= cap: return df
    vids = df.groupby("video").label.first()
    frac, keep = cap/len(df), []
    for lbl, grp in vids.groupby(vids):
        n = max(3, int(round(len(grp)*frac)))
        keep += list(pd.Series(grp.index).sample(min(n, len(grp)), random_state=seed))
    return df[df.video.isin(keep)].reset_index(drop=True)

def create_model(pretrained=True):
    return timm.create_model("efficientnet_b0", pretrained=pretrained, num_classes=1)


def load_checkpoint(model, path):
    ck = torch.load(path, map_location="cpu")
    for key in ("ema","model_ema","state_dict","model"):
        if isinstance(ck, dict) and key in ck and isinstance(ck[key], dict):
            ck = ck[key]; break
    ck = {k.replace("module.",""): v for k, v in ck.items()}
    # Phase 11 saves SingleNet / DualNet wrappers, whose keys carry a "net."
    # prefix. Without this strip every key mismatches, strict=False loads
    # NOTHING, and a randomly-initialised model gets scored as if it were real.
    if ck and all(k.startswith("net.") for k in ck):
        ck = {k[4:]: v for k, v in ck.items()}
    miss, unexp = model.load_state_dict(ck, strict=False)
    n_total = len(model.state_dict()); n_loaded = n_total - len(miss)
    if miss or unexp: print(f"  [load] missing={len(miss)} unexpected={len(unexp)}")
    if n_loaded < 0.9 * n_total:
        raise RuntimeError(
            f"{Path(path).name}: only {n_loaded}/{n_total} tensors loaded. "
            f"Refusing to score a model that did not load.")
    return model

print(f"\nTERM={TERM} | SMOKE={SMOKE} | epochs={EPOCHS} | device={DEVICE}")

# ===================== TERM 1: DATA AUDIT ============================
if TERM == 1:
    AUDIT = {}
    for name in ["celebdf", "ffpp", "dfdc"]:
        print("=" * 70)
        if PARSERS[name][0] is None:
            print(f"{name.upper()}: NOT FOUND"); continue
        df = build_manifest(name)
        fpv = len(df) / df.video.nunique()
        vl = df.groupby("video").label.first()
        print(f"{name.upper()}: {len(df):,} imgs | {df.video.nunique():,} videos "
              f"| {df.group.nunique():,} groups | {fpv:.1f} frames/video")
        print(f"  videos: real {int((vl==0).sum()):,} / fake {int((vl==1).sum()):,}")
        print("  OK" if fpv >= 5 else "  !! FAIL: parser not grouping frames")
        tr, va, te = split_by_video(df)
        AUDIT[name] = dict(images=len(df), videos=int(df.video.nunique()),
                           groups=int(df.group.nunique()), fpv=round(fpv,1))
    print(pd.DataFrame(AUDIT).T.to_string())
    print("\nTERM 1 done.")


## Cell 2 — Original dual-stream definitions only

Extracted from your supplied Phase 11 code. The only constructor change is `pretrained=False` by default, because the full trained checkpoint will be loaded. The corrected `std(dim=(2,3), keepdim=True)` is retained.

In [ ]:
# Definitions only: no old CAM/DANN/dual-stream training loops.
import math
import torch.nn.functional as F
from torchvision import transforms as T
MEAN, STD = (0.485,0.456,0.406), (0.229,0.224,0.225)
eval_tf = T.Compose([T.Resize((IMG_SIZE,IMG_SIZE)), T.ToTensor(), T.Normalize(MEAN,STD)])

def _dct_mat(n):
    k = torch.arange(n, dtype=torch.float64); i = torch.arange(n, dtype=torch.float64)
    D = torch.cos(math.pi * (2*i[None,:] + 1) * k[:,None] / (2*n))
    D *= math.sqrt(2.0/n); D[0] /= math.sqrt(2.0)
    return D.float()


class FAD(nn.Module):
    """Whole-image DCT -> additive learnable band masks -> inverse DCT.
    Bands come back in IMAGE space, so the frequency stream stays aligned
    with RGB and an ImageNet-pretrained stem still transfers."""
    def __init__(self, size=IMG_SIZE, edges=(0.0, 1/16, 1/8, 1.0)):
        super().__init__()
        self.n_bands = len(edges) - 1
        self.register_buffer("D", _dct_mat(size), persistent=False)
        u = torch.arange(size, dtype=torch.float32)
        r = (u[:,None] + u[None,:]) / (2*size - 2)
        self.register_buffer("base", torch.stack([
            ((r >= lo) & ((r < hi) if hi < 1.0 else (r <= hi))).float()
            for lo, hi in zip(edges[:-1], edges[1:])]), persistent=False)
        self.delta = nn.Parameter(torch.zeros(self.n_bands, size, size))
    def forward(self, x01):
        D = self.D
        X = D @ x01 @ D.transpose(0,1)
        m = self.base + torch.tanh(self.delta)
        X = X.unsqueeze(1) * m[None,:,None]
        return (D.transpose(0,1) @ X @ D).flatten(1,2)


def _feat_dim(m):
    was = m.training; m.eval()
    with torch.no_grad():
        f = m.forward_features(torch.zeros(1, 3, IMG_SIZE, IMG_SIZE,
                               device=next(m.parameters()).device))
    m.train(was)
    return f.shape[1]


class DualNet(nn.Module):
    """Takes ImageNet-NORMALISED input and returns (N,1), so every existing
    helper (predict / score / video_level) works on it unchanged.  The
    [0,1] image needed by the DCT is recovered internally."""
    def __init__(self, pretrained=False):
        super().__init__()
        self.rgb = create_model(pretrained=pretrained)
        assert hasattr(self.rgb, "forward_features"), \
            "create_model() must expose forward_features (timm backbone)"
        self.fad  = FAD()
        self.freq = timm.create_model("efficientnet_b0", pretrained=pretrained,
                                      num_classes=0, in_chans=3*self.fad.n_bands)
        self.register_buffer("mean", torch.tensor(MEAN).view(1,3,1,1))
        self.register_buffer("std",  torch.tensor(STD).view(1,3,1,1))
        d_r = _feat_dim(self.rgb); d_f = self.freq.num_features
        self.head      = nn.Sequential(nn.Dropout(0.2), nn.Linear(d_r + d_f, 1))
        self.head_rgb  = nn.Linear(d_r, 1)
        self.head_freq = nn.Linear(d_f, 1)
        self._aux, self._feat, self._pooled = None, None, None
    def forward(self, xn):
        fr = self.rgb.forward_features(xn); self._feat = fr
        zr = F.adaptive_avg_pool2d(fr, 1).flatten(1); self._pooled = zr
        x01 = (xn * self.std + self.mean).clamp(0, 1)
        with torch.autocast('cuda', enabled=False):
            fb = self.fad(x01.float())          # DC term is large: not fp16-safe
        fb = ((fb - fb.mean(dim=(2, 3), keepdim=True))
              / (fb.std(dim=(2, 3), keepdim=True) + 1e-5))
        zf = self.freq(fb.to(xn.dtype))
        self._aux = (self.head_rgb(zr).squeeze(1), self.head_freq(zf).squeeze(1))
        return self.head(torch.cat([zr, zf], 1))          # (N,1)


print("Original FAD and DualNet definitions loaded; pretrained downloads disabled.")


## Cell 3 — Connect original data and checkpoint

No separate CSV needed. Check the printed roots point to the actual face-crop datasets. Dataset labels and IDs use your original parsing code. This code cannot independently verify that folder-name labels or dataset snapshots are correct.

In [ ]:
# Build the pilot inputs from YOUR existing Cell 1 helpers.
from datetime import datetime, timezone

# Optional: explicitly correct any dataset root printed by Cell 1.
# Use the SAME crop dataset versions used to train the checkpoint.
ROOT_OVERRIDES = {
    # 'celebdf': '/kaggle/input/actual-celebdf-crops',
    # 'ffpp': '/kaggle/input/actual-ffpp-crops',
    # 'dfdc': '/kaggle/input/actual-dfdc-crops',
}
for name, path in ROOT_OVERRIDES.items():
    PARSERS[name] = (Path(path), PARSERS[name][1])
print('Image roots:', {name: str(pair[0]) for name,pair in PARSERS.items()})

# Prefer live original splits if Cell 2 previously built them. Otherwise replay
# your ORIGINAL parser + seed-42 split algorithm on the same dataset snapshot.
# A different set/version of images or identity groups can produce different
# splits despite the same seed. Do not call those original splits.
if 'SPLITS' in globals() and all(k in SPLITS for k in ['celebdf','ffpp','dfdc']):
    PILOT_SPLITS = SPLITS
    SPLIT_ORIGIN = 'existing in-memory original SPLITS'
else:
    PILOT_SPLITS = {}
    for name in ['celebdf','ffpp','dfdc']:
        original_df = build_manifest(name)
        PILOT_SPLITS[name] = split_by_video(original_df, seed=42)
    SPLIT_ORIGIN = 'original algorithm replay; assumes unchanged dataset snapshot'

pieces = []
for name in ['celebdf','ffpp','dfdc']:
    for split_name, frame in zip(['train','val','test'], PILOT_SPLITS[name]):
        part = frame.copy()
        part['dataset'] = name
        part['split'] = split_name
        pieces.append(part.rename(columns={'video':'video_id','group':'group_id'}))
MANIFEST = pd.concat(pieces, ignore_index=True)[
    ['path','dataset','video_id','group_id','label','split']
]
for column in ['video_id','group_id']:
    MANIFEST[column] = MANIFEST[column].astype(str)
print('Split source:', SPLIT_ORIGIN)
print(MANIFEST.groupby(['dataset','split']).size().rename('frames').to_string())

# Set a literal path if more than one matching checkpoint is attached.
CHECKPOINT_OVERRIDE = '/kaggle/input/datasets/kawsersikder/phase11-checkpoints/model_dualstream_loo_dfdc.pth'
if CHECKPOINT_OVERRIDE and Path(CHECKPOINT_OVERRIDE).is_file():
    CHECKPOINT = Path(CHECKPOINT_OVERRIDE)
else:   # pruned search: never lists the face-crop folders (a full walk took ~1 h in a commit)
    IMAGE_ROOTS = {str(PARSERS[n][0]) for n in ['celebdf', 'ffpp', 'dfdc']}
    matches = []
    for dp, dns, fns in os.walk('/kaggle/input'):
        matches += [Path(dp) / f for f in fns if f == 'model_dualstream_loo_dfdc.pth']
        if dp in IMAGE_ROOTS or dp.count(os.sep) >= 8 or any(f.lower().endswith(IMG_EXT) for f in fns):
            dns[:] = []
    matches.sort()
    print('Checkpoint candidates:', [str(p) for p in matches])
    assert len(matches) >= 1, 'Attach the phase11-checkpoints dataset (model_dualstream_loo_dfdc.pth).'
    CHECKPOINT = matches[0]
assert CHECKPOINT.is_file()
assert torch.cuda.is_available(), 'Enable a Kaggle GPU.'

PREPROCESS = eval_tf  # Exact deterministic transform from your original Cell 2.
CFG = dict(
    seed=42, device='cuda:0', amp=True,
    batch_size=32, workers=2, epochs=5, lr=1e-5,
    max_drop=0.35, train_per_source=9000,
    eval_frames_per_video=8, flip_tta=True, bootstrap=1000,
    split_origin=SPLIT_ORIGIN,
    output_dir='/kaggle/working/codec_pilot_' + datetime.now(timezone.utc).strftime('%Y%m%d_%H%M%S'),
)
print('Selected checkpoint:', CHECKPOINT)
print('GPU:', torch.cuda.get_device_name(0))


## Cell 4 — Pilot helper implementation

This is your supplied pilot helper cell. It defines functions only. It replaces the old `predict` name, so do not run old Phase 11/12 experiment cells after this.

In [ ]:
"""JPEG-inspired frequency-branch pilot. Requires the ORIGINAL dual-stream model.

No training runs on import.
The intervention is block-DCT masking BEFORE the existing `fad` module. It is
not H.264 simulation, a replacement FAD implementation, or full retraining.
"""
import copy
import hashlib
import io
import json
import math
import random
import time
from pathlib import Path

import numpy as np
import pandas as pd
from PIL import Image
import torch
from torch import nn
from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import roc_auc_score


JPEG_LUMA = [
    [16,11,10,16,24,40,51,61], [12,12,14,19,26,58,60,55],
    [14,13,16,24,40,57,69,56], [14,17,22,29,51,87,80,62],
    [18,22,37,56,68,109,103,77], [24,35,55,64,81,104,113,92],
    [49,64,78,87,103,121,120,101], [72,92,95,98,112,100,103,99],
]


def seed_all(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.benchmark = False
    torch.backends.cudnn.deterministic = True


class JPEGInspiredMask(nn.Module):
    """Same stochastic coefficient mask across channels/blocks of each image.

    Higher JPEG-table entries get higher drop probabilities. DC is preserved.
    No inverted-dropout rescaling: removing AC energy is the intervention.
    Uses an independent RNG so masking does not advance model dropout RNG.
    """
    def __init__(self, max_drop=0.35, seed=1042):
        super().__init__()
        if not 0 <= max_drop <= 1:
            raise ValueError('max_drop must lie in [0, 1]')
        n = torch.arange(8, dtype=torch.float32)
        k = n[:, None]
        basis = torch.cos(math.pi * (2*n + 1) * k / 16) * math.sqrt(2/8)
        basis[0] /= math.sqrt(2)
        q = torch.tensor(JPEG_LUMA, dtype=torch.float32)
        probability = max_drop * q / q.max()
        probability[0, 0] = 0
        self.register_buffer('basis', basis)
        self.register_buffer('probability', probability)
        self.max_drop = max_drop
        self.seed = seed
        self.generators = {}
        self.calls = 0

    def coefficients(self, x):
        b, c, h, w = x.shape
        if h % 8 or w % 8:
            raise ValueError('FAD input H/W must be divisible by 8')
        blocks = x.float().reshape(b,c,h//8,8,w//8,8).permute(0,1,2,4,3,5)
        return self.basis @ blocks @ self.basis.T

    def reconstruct(self, coeff):
        b, c, nh, nw, _, _ = coeff.shape
        blocks = self.basis.T @ coeff @ self.basis
        return blocks.permute(0,1,2,4,3,5).reshape(b,c,nh*8,nw*8)

    def forward(self, x):
        if not self.training or self.max_drop == 0:
            return x
        if x.ndim != 4 or not x.is_floating_point():
            raise ValueError('Expected floating BCHW spatial input to fad')
        self.calls += 1
        key = str(x.device)
        if key not in self.generators:
            self.generators[key] = torch.Generator(device=x.device).manual_seed(self.seed)
        # DCT stays in float32 even inside the model's AMP context.
        with torch.autocast(device_type=x.device.type, enabled=False):
            coeff = self.coefficients(x)
            u = torch.rand((len(x),1,1,1,8,8), device=x.device,
                           generator=self.generators[key])
            keep = (u >= self.probability).to(coeff.dtype)
            out = self.reconstruct(coeff * keep)
        return out.to(x.dtype)


class MaskedFrequencyModel(nn.Module):
    def __init__(self, base, max_drop=0.35, seed=1042):
        super().__init__()
        if not hasattr(base, 'fad') or not hasattr(base, 'rgb'):
            raise ValueError('Use original dual-stream model with .fad and .rgb modules')
        self.base = base
        self.mask = JPEGInspiredMask(max_drop, seed)
        self.handle = base.fad.register_forward_pre_hook(self._mask_input)

    def _mask_input(self, module, args):
        if len(args) != 1:
            raise ValueError('Expected fad(x); adapt hook explicitly for another signature')
        return (self.mask(args[0]),)

    def forward(self, x):
        return self.base(x)


def default_preprocess(image):
    """Replace if the ORIGINAL notebook used a different transform."""
    image = image.resize((224,224), Image.Resampling.BILINEAR)
    x = torch.from_numpy(np.asarray(image, dtype=np.float32).copy()).permute(2,0,1)/255
    return (x-torch.tensor([.485,.456,.406])[:,None,None]) / torch.tensor([.229,.224,.225])[:,None,None]


def default_logits(output):
    """Override explicitly if original forward returns a tuple/dict."""
    if not isinstance(output, torch.Tensor):
        raise TypeError('Set LOGITS_FN to select the fused binary LOGIT from model output')
    if output.ndim == 2 and output.shape[1] == 1:
        output = output[:,0]
    if output.ndim != 1:
        raise ValueError('Expected one binary logit per frame, not probabilities or two logits')
    return output


def validate_manifest(frame):
    required = {'path','dataset','video_id','group_id','label','split'}
    if required - set(frame):
        raise ValueError(f'Missing manifest columns: {required-set(frame)}')
    d = frame.copy()
    if d[list(required)].isna().any().any():
        raise ValueError('Manifest contains missing values')
    for col in ['dataset','video_id','group_id','split']:
        d[col] = d[col].astype(str).str.strip()
        if d[col].eq('').any():
            raise ValueError(f'Empty {col}')
    if not set(d.dataset) <= {'celebdf','ffpp','dfdc'}:
        raise ValueError('Use canonical dataset names: celebdf, ffpp, dfdc')
    if not set(d.split) <= {'train','val','test'}:
        raise ValueError('Use existing train/val/test assignments')
    if not set(d.label) <= {0,1}:
        raise ValueError('Labels must be 0=real, 1=fake')
    d['path'] = d.path.map(lambda p: str(Path(p).expanduser().resolve()))
    if d.path.duplicated().any():
        raise ValueError('Duplicate frame paths; use one row per file')
    if (d.groupby(['dataset','video_id']).label.nunique() > 1).any():
        raise ValueError('A video has conflicting labels')
    if (d.groupby(['dataset','video_id']).group_id.nunique() > 1).any():
        raise ValueError('A video has multiple identity/source groups')
    for key in ['video_id','group_id']:
        if (d.groupby(['dataset',key]).split.nunique() > 1).any():
            raise ValueError(f'{key} leakage across original splits')
    missing = [p for p in d.path if not Path(p).is_file()]
    if missing:
        raise FileNotFoundError(f'{len(missing)} missing frames; first: {missing[0]}')
    # IDs are namespaced by corpus; cross-corpus duplicate identities still need
    # the project's existing identity audit. This is not an image deduplication tool.
    return d


def diverse_cap(d, cap, seed):
    # Shuffle frames, then take one per video before taking a second, etc.
    d = d.sample(frac=1, random_state=seed).copy()
    d['_round'] = d.groupby(['dataset','video_id']).cumcount()
    return d.sort_values('_round', kind='stable').head(cap).drop(columns='_round')


def select_frames(d, cfg):
    source = d[d.dataset.isin(['celebdf','ffpp'])]
    train = pd.concat([diverse_cap(source[(source.dataset==ds)&(source.split=='train')],
                                    cfg['train_per_source'], cfg['seed'])
                       for ds in ['celebdf','ffpp']])
    val = source[source.split=='val'].sort_values('path').groupby(
        ['dataset','video_id'], group_keys=False).head(cfg['eval_frames_per_video'])
    # All DFDC videos are external to the checkpoint's training sources.
    test = d[d.dataset=='dfdc'].sort_values('path').groupby(
        ['dataset','video_id'], group_keys=False).head(cfg['eval_frames_per_video'])
    for name, part in [('train',train),('val',val),('test',test)]:
        if part.empty or part.label.nunique()!=2:
            raise ValueError(f'{name} must contain both real and fake samples')
        if name != 'test':
            for ds in ['celebdf','ffpp']:
                if part[part.dataset==ds].label.nunique()!=2:
                    raise ValueError(f'{name}/{ds} needs both classes')
    return train.reset_index(drop=True), val.reset_index(drop=True), test.reset_index(drop=True)


class FrameDataset(Dataset):
    def __init__(self, frame, preprocess, quality=None):
        self.frame = frame.reset_index(drop=True)
        self.preprocess = preprocess
        self.quality = quality

    def __len__(self):
        return len(self.frame)

    def __getitem__(self, index):
        row = self.frame.iloc[index]
        with Image.open(row.path) as im:
            image = im.convert('RGB')
        if self.quality is not None:
            buf = io.BytesIO()
            image.save(buf, format='JPEG', quality=self.quality, subsampling=2)
            buf.seek(0)
            with Image.open(buf) as decoded:
                image = decoded.convert('RGB')
        return self.preprocess(image), float(row.label), index


def loader(frame, preprocess, cfg, train=False, quality=None):
    return DataLoader(FrameDataset(frame,preprocess,quality),
                      batch_size=cfg['batch_size'], shuffle=train,
                      num_workers=cfg['workers'], pin_memory=cfg['device'].startswith('cuda'),
                      generator=torch.Generator().manual_seed(cfg['seed']))


def video_scores(frames, probabilities):
    d = frames[['dataset','video_id','group_id','label']].copy()
    d['probability'] = probabilities
    return d.groupby(['dataset','video_id','group_id','label'], as_index=False).probability.mean()


@torch.no_grad()
def predict(model, frames, preprocess, logits_fn, cfg, quality=None):
    model.eval()
    probability = np.empty(len(frames), dtype=np.float64)
    for x, _, indices in loader(frames,preprocess,cfg,quality=quality):
        x = x.to(cfg['device'], non_blocking=True)
        with torch.autocast(device_type=cfg['device'].split(':')[0], enabled=cfg['amp']):
            p = logits_fn(model(x)).float().sigmoid()
            if cfg['flip_tta']:
                p = (p + logits_fn(model(x.flip(-1))).float().sigmoid())/2
        probability[indices.numpy()] = p.cpu().numpy()
    if not np.isfinite(probability).all():
        raise FloatingPointError('Non-finite predictions')
    return video_scores(frames, probability)


def macro_auc(predictions):
    return float(np.mean([roc_auc_score(g.label,g.probability)
                          for _,g in predictions.groupby('dataset')]))


def file_sha256(path):
    h = hashlib.sha256()
    with open(path,'rb') as f:
        for block in iter(lambda:f.read(1024*1024),b''):
            h.update(block)
    return h.hexdigest()


def load_base(template, checkpoint, device):
    model = copy.deepcopy(template).cpu()
    state = torch.load(checkpoint,map_location='cpu',weights_only=True)
    if not isinstance(state,dict) or 'rgb.conv_stem.weight' not in state:
        raise ValueError('Expected original plain state_dict; unwrap explicitly if needed')
    model.load_state_dict(state,strict=True)
    return model.to(device)


def fit_arm(template, checkpoint, train, val, preprocess, logits_fn, cfg, name, out):
    seed_all(cfg['seed'])
    base = load_base(template,checkpoint,cfg['device'])
    for p in base.parameters():
        p.requires_grad_(True)
    model = MaskedFrequencyModel(base,cfg['max_drop'],cfg['seed']+1000).to(cfg['device']) if name=='masked' else base
    opt = torch.optim.AdamW(model.parameters(),lr=cfg['lr'],weight_decay=1e-4)
    scaler = torch.amp.GradScaler('cuda',enabled=cfg['amp'])
    pos_weight = torch.tensor([(train.label==0).sum()/(train.label==1).sum()],device=cfg['device'])
    criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight)
    dl = loader(train,preprocess,cfg,train=True)
    best, rows = -float('inf'), []
    start = time.perf_counter()
    for epoch in range(1,cfg['epochs']+1):
        model.train()
        total, n, skipped = 0., 0, 0
        for x,y,_ in dl:
            x,y = x.to(cfg['device']),y.float().to(cfg['device'])
            opt.zero_grad(set_to_none=True)
            with torch.autocast(device_type=cfg['device'].split(':')[0],enabled=cfg['amp']):
                loss = criterion(logits_fn(model(x)),y)
            if not torch.isfinite(loss):
                raise FloatingPointError(f'{name}: non-finite training loss')
            scaler.scale(loss).backward()
            scaler.unscale_(opt)
            norm = nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            if not torch.isfinite(norm):
                skipped += 1   # fp16 overflow: GradScaler skips this step and lowers its scale
            scaler.step(opt)
            scaler.update()
            total += loss.item()*len(x)
            n += len(x)
        if skipped > 0.05 * len(dl):
            raise FloatingPointError(f'{name}: {skipped} of {len(dl)} steps overflowed; not just GradScaler warm-up')
        auc = macro_auc(predict(model,val,preprocess,logits_fn,cfg))
        elapsed = time.perf_counter()-start
        row = dict(arm=name,epoch=epoch,loss=total/n,source_val_auc=auc,elapsed_seconds=elapsed)
        rows.append(row)
        pd.DataFrame(rows).to_csv(out/f'{name}_training.csv',index=False)
        if auc > best:
            best = auc
            # Persist only the original architecture's weights, without the hook.
            torch.save(base.state_dict(),out/f'{name}_best.pth')
        print(f'{name}: epoch {epoch}/{cfg["epochs"]}, loss={total/n:.4f}, source val AUC={auc:.4f}, {elapsed/60:.1f} min',flush=True)
    if name=='masked' and model.mask.calls==0:
        raise RuntimeError('Mask hook never executed')
    del model,base,opt
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    return rows


def paired_cluster_interval(control, masked, n_boot=1000, seed=42):
    keys = ['dataset','video_id','group_id','label']
    d = control.merge(masked,on=keys,validate='one_to_one',suffixes=('_control','_masked'))
    if len(d)!=len(control) or len(d)!=len(masked):
        raise ValueError('The two arms did not evaluate exactly the same videos')
    delta = roc_auc_score(d.label,d.probability_masked)-roc_auc_score(d.label,d.probability_control)
    groups = list(d.groupby(['dataset','group_id']).indices.values())
    if len(groups)<2:
        raise ValueError('Need multiple independent evaluation groups')
    rng = np.random.default_rng(seed)
    samples=[]
    for _ in range(n_boot):
        indices=np.concatenate([groups[i] for i in rng.integers(0,len(groups),len(groups))])
        b=d.iloc[indices]
        if b.label.nunique()==2:
            samples.append(roc_auc_score(b.label,b.probability_masked)-roc_auc_score(b.label,b.probability_control))
    if len(samples)<0.9*n_boot:
        raise ValueError('Too few valid bootstrap samples; inspect class/group balance')
    lo,hi=np.quantile(samples,[.025,.975])
    return dict(delta_auc=float(delta),ci_low=float(lo),ci_high=float(hi),valid_bootstraps=len(samples))


def run_pilot(template, checkpoint, manifest, preprocess, logits_fn, cfg):
    out=Path(cfg['output_dir'])
    out.mkdir(parents=True,exist_ok=False)  # Never overwrite an earlier experiment.
    start=time.perf_counter()
    d=validate_manifest(manifest)
    train,val,test=select_frames(d,cfg)
    for name,part in [('train',train),('val',val),('test',test)]:
        part.to_csv(out/f'{name}_manifest.csv',index=False)
    metadata=dict(config=cfg,checkpoint_sha256=file_sha256(checkpoint),
                  torch_version=torch.__version__,numpy_version=np.__version__,
                  gpu=torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU',
                  protocol='single-seed matched continuation; source-only checkpoint selection; JPEG crop recompression',
                  status='running')
    (out/'run_metadata.json').write_text(json.dumps(metadata,indent=2))
    summary=d.groupby(['dataset','split']).agg(frames=('path','size'),videos=('video_id','nunique'))
    summary.to_csv(out/'input_counts.csv')
    print('Selected frames:',{'train':len(train),'val':len(val),'dfdc_test':len(test)},flush=True)
    print('No DFDC scores are used to choose epochs, mask strength, or checkpoints.',flush=True)
    history=[]
    for name in ['control','masked']:
        history.extend(fit_arm(template,checkpoint,train,val,preprocess,logits_fn,cfg,name,out))
    pd.DataFrame(history).to_csv(out/'training_history.csv',index=False)
    records, predictions = [], {}
    for arm,ckpt in [('original',checkpoint),('control',out/'control_best.pth'),('masked',out/'masked_best.pth')]:
        model=load_base(template,ckpt,cfg['device'])
        for condition,q in [('clean',None),('JPEG q50',50),('JPEG q20',20)]:
            pred=predict(model,test,preprocess,logits_fn,cfg,q)
            pred.to_csv(out/f'predictions_{arm}_{condition.replace(" ","_")}.csv',index=False)
            predictions[(arm,condition)]=pred
            records.append(dict(arm=arm,condition=condition,auc=macro_auc(pred),
                                videos=len(pred),real_videos=int((pred.label==0).sum()),
                                fake_videos=int((pred.label==1).sum())))
            pd.DataFrame(records).to_csv(out/'results.csv',index=False)
            print(records[-1],flush=True)
        del model
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
    deltas=[]
    for condition in ['clean','JPEG q50','JPEG q20']:
        row=paired_cluster_interval(predictions[('control',condition)],predictions[('masked',condition)],cfg['bootstrap'],cfg['seed'])
        deltas.append(dict(condition=condition,**row))
    result=pd.DataFrame(records)
    change=pd.DataFrame(deltas)
    change.to_csv(out/'paired_deltas.csv',index=False)
    make_plots(result,pd.DataFrame(history),out)
    metadata.update(status='complete',total_wall_hours=(time.perf_counter()-start)/3600)
    (out/'run_metadata.json').write_text(json.dumps(metadata,indent=2))
    table=result.pivot(index='condition',columns='arm',values='auc').reindex(['clean','JPEG q50','JPEG q20'])
    table=table.join(change.set_index('condition')[['delta_auc','ci_low','ci_high']])
    table.to_csv(out/'faculty_table.csv')
    print('\nFaculty table (delta = masked - matched control):\n',table.round(4).to_string())
    print(f'Finished in {metadata["total_wall_hours"]:.2f} wall hours. Files: {out}')
    return table


def make_plots(result,history,out):
    import matplotlib.pyplot as plt
    conditions=['clean','JPEG q50','JPEG q20']
    fig,ax=plt.subplots(figsize=(8,4.5))
    for arm in ['original','control','masked']:
        values=result[result.arm==arm].set_index('condition').loc[conditions,'auc']
        ax.plot(conditions,values,marker='o',label=arm)
    ax.axhline(.5,color='gray',ls='--',lw=1)
    ax.set(ylabel='Video-level ROC-AUC',ylim=(0,1),title='DFDC held out: JPEG recompression pilot')
    ax.legend();fig.tight_layout();fig.savefig(out/'robustness_curve.png',dpi=180);plt.close(fig)
    fig,axes=plt.subplots(1,2,figsize=(10,4))
    for arm,g in history.groupby('arm'):
        axes[0].plot(g.epoch,g.loss,marker='o',label=arm)
        axes[1].plot(g.epoch,g.source_val_auc,marker='o',label=arm)
    axes[0].set(xlabel='Epoch',ylabel='Training BCE')
    axes[1].set(xlabel='Epoch',ylabel='Source validation macro AUC')
    axes[1].legend();fig.tight_layout();fig.savefig(out/'training_curves.png',dpi=180);plt.close(fig)


def unit_tests():
    seed_all(7)
    x=torch.randn(2,3,16,24,requires_grad=True)
    m=JPEGInspiredMask(.7)
    c=m.coefficients(x)
    assert torch.allclose(m.reconstruct(c),x,atol=5e-6), 'DCT round trip'
    m.eval();assert m(x) is x, 'Evaluation must disable masking'
    m.train();y=m(x)
    assert y.shape==x.shape and torch.isfinite(y).all()
    assert not torch.allclose(x,y), 'Training mask must change AC content'
    assert torch.allclose(m.coefficients(y)[...,0,0],c[...,0,0],atol=1e-5), 'DC changed'
    assert m.coefficients(y).square().sum() <= c.square().sum()+1e-3, 'Energy increased'
    y.square().mean().backward();assert torch.isfinite(x.grad).all(), 'Gradient failure'
    assert m.probability[7,7] > m.probability[0,1] > m.probability[0,0]
    a,b=JPEGInspiredMask(.7,8),JPEGInspiredMask(.7,8)
    assert torch.equal(a(x.detach()),b(x.detach())), 'Mask RNG not reproducible'
    rng=torch.random.get_rng_state();a(x.detach())
    assert torch.equal(rng,torch.random.get_rng_state()), 'Mask changed global RNG'
    class Toy(nn.Module):
        def __init__(self):
            super().__init__();self.rgb=nn.Identity();self.fad=nn.Identity()
        def forward(self,x):
            return self.rgb(x),self.fad(x)
    w=MaskedFrequencyModel(Toy(),.7)
    rgb,freq=w(x.detach())
    assert torch.equal(rgb,x.detach()), 'RGB branch was changed'
    assert not torch.equal(freq,x.detach()), 'Frequency branch was unchanged'
    w.eval();rgb,freq=w(x.detach());assert torch.equal(rgb,freq)
    print('PASS: DCT, shape, DC preservation, energy, gradients, RNG, branch isolation, eval identity')
    return True


def preflight(template,checkpoint,frames,preprocess,logits_fn,cfg):
    """Checks actual architecture, strict loading, finite gradients, eval identity."""
    unit_tests()
    seed_all(cfg['seed'])
    model=load_base(template,checkpoint,cfg['device'])
    x,_,_=next(iter(loader(frames.head(2),preprocess,cfg)))
    x=x.to(cfg['device'])
    model.eval()
    with torch.no_grad():
        original=logits_fn(model(x)).detach().clone()
    wrapped=MaskedFrequencyModel(model,cfg['max_drop'],cfg['seed']+1000).to(cfg['device'])
    wrapped.eval()
    with torch.no_grad():
        disabled=logits_fn(wrapped(x))
    assert torch.allclose(original,disabled,atol=1e-6,rtol=1e-5), 'Eval logits changed with disabled mask'
    for p in wrapped.parameters():
        p.requires_grad_(True)
    wrapped.train()
    with torch.autocast(device_type=cfg['device'].split(':')[0],enabled=cfg['amp']):
        values=logits_fn(wrapped(x))
        loss=nn.functional.binary_cross_entropy_with_logits(values,torch.zeros_like(values))
    loss.backward()
    assert wrapped.mask.calls>0, 'FAD hook did not run'
    grads=[p.grad for p in wrapped.parameters() if p.grad is not None]
    assert grads and all(torch.isfinite(g).all() for g in grads), 'Invalid actual-model gradients'
    print('PASS: strict checkpoint load, actual-model eval identity, active FAD hook, finite backward pass')
    del model,wrapped
    if torch.cuda.is_available():
        torch.cuda.empty_cache()


def data_and_statistics_tests():
    """Synthetic tests only; creates temporary tiny images, never trains a model."""
    import tempfile
    with tempfile.TemporaryDirectory() as root:
        paths=[]
        for i in range(4):
            p=Path(root)/f'{i}.png'
            Image.new('RGB',(16,16),(30*i,60,100)).save(p)
            paths.append(str(p))
        frame=pd.DataFrame(dict(path=paths,dataset=['ffpp']*4,
                                video_id=['v0','v1','v2','v3'],
                                group_id=['g0','g1','g2','g3'],label=[0,1,0,1],
                                split=['train','train','val','val']))
        validate_manifest(frame)
        leaked=frame.copy();leaked.loc[2,'group_id']='g0'
        try:
            validate_manifest(leaked)
        except ValueError as e:
            assert 'leakage' in str(e)
        else:
            raise AssertionError('Identity leakage was accepted')
        dup=frame.copy();dup.loc[1,'path']=dup.loc[0,'path']
        try:
            validate_manifest(dup)
        except ValueError as e:
            assert 'Duplicate' in str(e)
        else:
            raise AssertionError('Duplicate paths were accepted')
        image,_,_=FrameDataset(frame,default_preprocess,20)[0]
        assert image.shape==(3,224,224) and torch.isfinite(image).all()
    rows=pd.DataFrame(dict(dataset=['dfdc']*20,video_id=[str(i) for i in range(20)],
                           group_id=[str(i) for i in range(20)],label=[0,1]*10,
                           probability=[.2,.8]*10))
    interval=paired_cluster_interval(rows,rows,n_boot=100)
    assert interval['delta_auc']==interval['ci_low']==interval['ci_high']==0
    repeated=pd.concat([rows,rows],ignore_index=True)
    pooled=video_scores(repeated,repeated.probability.to_numpy())
    assert len(pooled)==20
    assert macro_auc(pooled)==1
    print('PASS: split-leakage rejection, duplicate rejection, JPEG loader, video pooling, paired bootstrap')
    return True


## Cell 5 — Synthetic tests, data audit, and real-model preflight

All tests must pass. The preflight uses a disposable model copy. It verifies checkpoint compatibility, unchanged evaluation behavior, active masking, and a finite backward pass. It does not train or alter the original checkpoint.

In [ ]:
unit_tests()
data_and_statistics_tests()

AUDITED = validate_manifest(MANIFEST)
TRAIN, VAL, TEST = select_frames(AUDITED, CFG)
for name, frame in [('TRAIN',TRAIN),('SOURCE VAL',VAL),('DFDC TEST',TEST)]:
    print('\n' + name)
    display(frame.groupby(['dataset','label']).agg(frames=('path','size'),videos=('video_id','nunique')))
assert not TRAIN.dataset.eq('dfdc').any()
assert not VAL.dataset.eq('dfdc').any()

# A fresh template avoids copying old non-leaf activation caches (_feat/_aux).
# All trained weights are loaded strictly from CHECKPOINT, not downloaded.
MODEL_TEMPLATE = DualNet(pretrained=False).cpu()
LOGITS_FN = default_logits  # Verified original DualNet.forward returns (N,1).
preflight(MODEL_TEMPLATE, CHECKPOINT, TRAIN, PREPROCESS, LOGITS_FN, CFG)
print('READY: preflight passed. Next cell starts the two short training runs.')


## Cell 6 — Run the pilot

Ordinary fine-tuning and masked fine-tuning start from the same checkpoint, use the same five-epoch schedule, and select checkpoints on source validation only. DFDC clean/q50/q20 is evaluated afterward. Histories/checkpoints are saved each epoch. No hard deadline is enforced. A fresh output directory is required for a new run.

In [ ]:
FACULTY_TABLE = run_pilot(
    MODEL_TEMPLATE, CHECKPOINT, AUDITED, PREPROCESS, LOGITS_FN, CFG
)
display(FACULTY_TABLE.round(4))


## Cell 7 — Faculty outputs and download

Include `faculty_table.csv`, `robustness_curve.png`, and `training_curves.png`, plus video/class counts, configuration, and measured wall time. Keep all per-video predictions and the ZIP as supporting evidence. Compare masked versus control on the same frames, not directly with the older report AUC. The eight-frame pilot sampling differs from the original up-to-32-frame evaluation.

The paired bootstrap uses the original group IDs; for DFDC those are videos. Its interval describes sampling uncertainty, not seed variability. If it crosses zero, the improvement is inconclusive. Do not claim H.264 robustness.

In [ ]:
from IPython.display import Image as DisplayImage, FileLink, display
import shutil
RESULT_DIR = Path(CFG['output_dir'])
metadata = json.loads((RESULT_DIR/'run_metadata.json').read_text())
assert metadata['status']=='complete', 'Do not present an incomplete run as final.'
display(pd.read_csv(RESULT_DIR/'faculty_table.csv').round(4))
display(DisplayImage(filename=str(RESULT_DIR/'robustness_curve.png')))
display(DisplayImage(filename=str(RESULT_DIR/'training_curves.png')))
print('Elapsed pipeline hours:', round(metadata['total_wall_hours'],3))
print('GPU:', metadata['gpu'])
archive = shutil.make_archive(str(RESULT_DIR), 'zip', root_dir=RESULT_DIR)
display(FileLink(archive))
